# MET Kinase Zero-Shot ESM Scoring: Build + Validate Against Estevam et al. 2025

**Purpose:** Build our own ESM-1b (and ESM-2, multiple sizes) zero-shot mutation-sensitivity scorer, and validate it by checking whether it reproduces the published correlation from Estevam et al. 2025 (eLife 13:RP101882, doi.org/10.7554/eLife.101882.3) between ESM-1b zero-shot score and real drug-treated MET kinase-domain resistance fitness (their reported r ~ 0.28 pooled across 9 inhibitors).

**Why this notebook exists:** their processed data (mutation x drug fitness table with a precomputed ESM-LLR `score` column) is public at `github.com/fraser-lab/MET_kinase_Inhibitor_DMS` (MIT license), but their exact ESM-1b scoring script is not checked into that repo. Rather than trust a black-box precomputed column, this notebook builds a standard, from-scratch zero-shot scorer (following the masked-marginal log-likelihood-ratio method of Meier et al., NeurIPS 2021) and checks that it reproduces their numbers on MET, before trusting the same pipeline on EGFR and ABL1 later in the study.

**Data attribution:** WT/mutant fitness and drug-treatment data from Estevam GO et al., "Mapping kinase domain resistance mechanisms for the MET receptor tyrosine kinase via deep mutational scanning," eLife 2025;13:RP101882. Code repo: https://github.com/fraser-lab/MET_kinase_Inhibitor_DMS (MIT License).

**Run this on Kaggle with GPU + internet enabled.** ESM-2 3B needs a GPU with >=16GB memory (Kaggle's P100/T4x2 both work; 3B is the largest size attempted here, 15B is a stretch goal for a rented cloud GPU, not this notebook).

**Method (masked-marginal LLR, Meier et al. 2021):** for each position, mask the wild-type residue, run one forward pass, take the log-softmax over the vocabulary at that position, then score = log P(mutant aa) - log P(wildtype aa). This requires only one forward pass per *position* (not per mutation), since a single masked forward pass gives log-probabilities for all 20 amino acids at once.

## 1. Setup

In [ ]:
!pip install -q fair-esm
!rm -rf MET_kinase_Inhibitor_DMS
!git clone --depth 1 https://github.com/fraser-lab/MET_kinase_Inhibitor_DMS.git

import torch
import esm
import pandas as pd
import numpy as np
from scipy import stats
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Reconstruct the exact WT construct sequence from their own data

Rather than assume a UniProt numbering (their DMS construct is numbered 1-287, not full-length MET UniProt numbering), we reconstruct the reference sequence directly from the `wildtype` column of their own `WT_rosace_effect_all.tsv` file — this guarantees an exact match to whatever construct they actually screened, no numbering-convention risk.

In [ ]:
wt_df = pd.read_csv('MET_kinase_Inhibitor_DMS/WT_rosace_effect_all.tsv', sep='\t')
wt_map = wt_df[['position', 'wildtype']].drop_duplicates().sort_values('position')

positions = sorted(wt_map['position'].unique())
assert positions == list(range(1, len(positions) + 1)), 'position numbering has gaps - stop and investigate before proceeding'

WT_SEQ = ''.join(wt_map.set_index('position').loc[positions, 'wildtype'])
print('Reconstructed WT MET kinase-domain construct length:', len(WT_SEQ))
print(WT_SEQ)

## 3. Load their mutation panel + published ESM-LLR scores (for validation)

In [ ]:
mut_panel = feat_df[['pos_mut', 'pos', 'score']].drop_duplicates(subset=['pos_mut']).reset_index(drop=True)

def parse_pos_mut(pos_mut_str):
    m = re.match(r'^(\d+)(\w)$', pos_mut_str)
    pos = int(m.group(1))
    mut_aa = m.group(2)
    wt_aa = WT_SEQ[pos - 1]
    return pos, wt_aa, mut_aa

parsed = mut_panel['pos_mut'].apply(parse_pos_mut)
mut_panel['pos'] = parsed.apply(lambda t: t[0])
mut_panel['wt_aa'] = parsed.apply(lambda t: t[1])
mut_panel['mut_aa'] = parsed.apply(lambda t: t[2])

# sanity check: does our reconstructed WT sequence match the wt residue implied by their own wildtype column?
check = wt_df[['position','wildtype']].drop_duplicates().set_index('position')
mismatches = 0
for _, row in mut_panel.iterrows():
    expected = check.loc[row['pos'], 'wildtype']
    if row['wt_aa'] != expected:
        mismatches += 1
print(f'WT-residue mismatches between reconstructed sequence and their wildtype column: {mismatches} / {len(mut_panel)}')
assert mismatches == 0, 'Reference sequence reconstruction is wrong - stop and investigate before running any model'

print(f'{len(mut_panel)} unique mutations across {mut_panel["pos"].nunique()} positions loaded and verified')
mut_panel.head()

## 4. Zero-shot masked-marginal scoring function

One masked forward pass per **position** (not per mutation) — the masked position's output logits give log-probabilities for all 20 amino acids simultaneously, which covers every mutation observed at that position in one pass.

In [ ]:
def compute_masked_marginal_scores(model, alphabet, wt_seq, positions_to_score, device):
    """
    Returns a dict: {position (1-indexed) -> {amino_acid: log_prob}}
    following Meier et al. 2021's masked-marginal method.
    """
    batch_converter = alphabet.get_batch_converter()
    model.eval()
    _, _, base_tokens = batch_converter([('wt', wt_seq)])
    base_tokens = base_tokens.to(device)

    log_probs_by_position = {}
    with torch.no_grad():
        for pos in positions_to_score:
            tok_idx = pos  # +1 for BOS token offset is handled by alphabet; ESM batch_converter prepends <cls>
            tokens = base_tokens.clone()
            tokens[0, pos] = alphabet.mask_idx  # position `pos` in 1-indexed sequence == index `pos` in token tensor (index 0 is <cls>)
            out = model(tokens, repr_layers=[], return_contacts=False)
            logits = out['logits'][0, pos]  # vocab logits at the masked position
            log_probs = torch.log_softmax(logits, dim=-1)
            log_probs_by_position[pos] = {
                aa: log_probs[alphabet.get_idx(aa)].item()
                for aa in 'ACDEFGHIKLMNPQRSTVWY'
            }
    return log_probs_by_position

## 5. Run ESM-1b and score all mutations, compare to their published scores

In [ ]:
print('Loading ESM-1b (650M) ...')
model_1b, alphabet_1b = esm.pretrained.esm1b_t33_650M_UR50S()
model_1b = model_1b.to(device)

unique_positions = sorted(mut_panel['pos'].unique())
print(f'Scoring {len(unique_positions)} positions with ESM-1b (one forward pass each)...')
logprobs_1b = compute_masked_marginal_scores(model_1b, alphabet_1b, WT_SEQ, unique_positions, device)

def llr_score(row, logprobs):
    lp = logprobs[row['pos']]
    return lp[row['mut_aa']] - lp[row['wt_aa']]

mut_panel['our_esm1b_score'] = mut_panel.apply(lambda r: llr_score(r, logprobs_1b), axis=1)

rho, p = stats.spearmanr(mut_panel['our_esm1b_score'], mut_panel['score'])
print(f'\\nValidation: Spearman(our from-scratch ESM-1b LLR, their published score) = {rho:.4f} (p={p:.2e})')
print('This should be very high (>0.95) if our scoring pipeline is implemented correctly -')
print('both are the same masked-marginal method on the same model, so near-perfect agreement is expected,')
print('not just directional agreement. A low correlation here means something in the reimplementation is wrong -')
print('stop and debug before trusting this pipeline on EGFR/ABL1.')

## 6. Re-run the drug-resistance correlation with OUR scores (not their precomputed ones)

This is the actual replication step: does our own independently-built scorer, not just their precomputed column, reproduce the r~0.28 pooled / per-drug correlations against real drug-treated fitness?

In [ ]:
merged = feat_df.merge(mut_panel[['pos_mut', 'our_esm1b_score']], on='pos_mut', how='left')

print('Per-drug Spearman(our ESM-1b LLR, drug-treated fitness):')
results = []
for drug in sorted(merged['key'].unique()):
    sub = merged[merged['key'] == drug].dropna(subset=['our_esm1b_score', 'mean'])
    rho, p = stats.spearmanr(sub['our_esm1b_score'], sub['mean'])
    results.append({'drug': drug, 'n': len(sub), 'our_spearman_rho': rho, 'p': p})
    print(f'  {drug:6s}  n={len(sub):6d}  our_rho={rho:.3f}')

overall = merged.dropna(subset=['our_esm1b_score', 'mean'])
rho, p = stats.spearmanr(overall['our_esm1b_score'], overall['mean'])
print(f'\\nPooled (our from-scratch scorer): n={len(overall)}, rho={rho:.4f}')
print('Compare to: paper-reported r~0.28; our own precomputed-column check earlier this session gave pooled rho=0.274')

results_df = pd.DataFrame(results)
results_df.to_csv('met_esm1b_per_drug_validation.csv', index=False)

## 7. ESM-2, multiple sizes (150M / 650M / 3B) - the model-scale comparison

This is finalized-plan point 2: does model scale/generation improve the drug-resistance-specific correlation at all, or is the weak correlation a fundamental limitation of the zero-shot approach regardless of model size?

Run 150M and 650M first (fast); only run 3B if GPU memory allows (~12GB+ free) - it is the slowest and most memory-hungry step in this notebook.

In [ ]:
esm2_checkpoints = {
    'esm2_150M': esm.pretrained.esm2_t30_150M_UR50D,
    'esm2_650M': esm.pretrained.esm2_t33_650M_UR50D,
    'esm2_3B':   esm.pretrained.esm2_t36_3B_UR50D,
}

all_scores = mut_panel[['pos_mut', 'pos', 'wt_aa', 'mut_aa', 'score', 'our_esm1b_score']].copy()
all_scores = all_scores.rename(columns={'score': 'estevam_published_esm1b_score'})

for name, loader in esm2_checkpoints.items():
    print(f'\\nLoading {name} ...')
    try:
        model, alphabet = loader()
        model = model.to(device)
        logprobs = compute_masked_marginal_scores(model, alphabet, WT_SEQ, unique_positions, device)
        all_scores[f'{name}_score'] = all_scores.apply(
            lambda r: logprobs[r['pos']][r['mut_aa']] - logprobs[r['pos']][r['wt_aa']], axis=1
        )
        del model
        torch.cuda.empty_cache()
        print(f'{name} done.')
    except RuntimeError as e:
        print(f'{name} FAILED (likely out of GPU memory): {e}')
        print('Skip and continue - rerun this checkpoint alone on a larger GPU if needed.')

all_scores.to_csv('met_all_esm_scores.csv', index=False)
print('\\nSaved met_all_esm_scores.csv')
all_scores.head()

In [ ]:
merged_all = feat_df.merge(all_scores, on='pos_mut', how='left')

score_cols = [c for c in all_scores.columns if c.endswith('_score')]
print('Pooled Spearman(model score, drug-treated fitness) by model:')
for col in score_cols:
    sub = merged_all.dropna(subset=[col, 'mean'])
    if len(sub) == 0:
        continue
    rho, p = stats.spearmanr(sub[col], sub['mean'])
    print(f'  {col:22s}  n={len(sub):6d}  rho={rho:.4f}')

print('\\nThis table is the core evidence for finalized-plan point 2 (does ESM-2 scale beat ESM-1b for')
print('drug-specific resistance) on MET. The same pipeline (Sections 4-7) should be re-run unchanged on')
print('EGFR and ABL1 sequences/mutation panels once the paired ChEMBL WT/mutant data is ready.')

## Next steps (outside this notebook)

1. Check Section 5's validation correlation is high (>0.95) before trusting anything downstream - if it isn't, the masked-marginal reimplementation has a bug (most likely culprit: token-index offset, or alphabet AA ordering) and must be fixed first.
2. Once validated, reuse `compute_masked_marginal_scores()` unchanged for EGFR (T790M/C797S/L858R) and ABL1 (M244V/G250E/Y253F/H/E255K/V/T315I/M351T/F359V/I) after pulling paired WT/mutant ChEMBL data.
3. Add the non-learned baselines (BLOSUM62, conservation, structural distance-to-pocket) from finalized-plan point 7 for comparison against the ESM scores computed here.
4. Bootstrap confidence intervals + paired statistical comparison between model sizes (finalized-plan point 8) using the per-mutation scores saved in `met_all_esm_scores.csv`.